# Notebook 3 — Churn Prediction Model
**What this does:**  
- Trains a Random Forest model to predict which subscribers will churn  
- Shows what factors drive churn most  
- Scores every active subscriber with a churn probability  
- Saves the risk scores as a CSV for Power BI  

**Why ML?**  
EDA tells us WHAT happened. ML tells us WHAT WILL HAPPEN NEXT.

In [ ]:
# ── CELL 1 ── Imports ────────────────────────────────────────────────────────
# sklearn = scikit-learn, Python's main machine learning library
# RandomForestClassifier → our main model
# train_test_split       → splits data into training and test sets
# roc_auc_score          → measures model quality (0.5=random, 1.0=perfect)
# classification_report  → shows precision, recall, F1 scores
# confusion_matrix       → 2x2 table of correct vs wrong predictions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

FILE_PATH  = r'E:\Mavenflix project\data\Subscription Cohort Analysis Data.csv'
OUTPUT_DIR = r'E:\Mavenflix project\outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('All imports loaded successfully!')

In [ ]:
# ── CELL 2 ── Load data & create features ────────────────────────────────────
# WHY FEATURE ENGINEERING?
# ML models only understand numbers — not dates, not text like "Yes"/"No".
# We convert everything into numbers the model can learn from.
# Each new column we create is called a "feature".

df = pd.read_csv(FILE_PATH)
df['created_date']  = pd.to_datetime(df['created_date'])
df['canceled_date'] = pd.to_datetime(df['canceled_date'])
SNAPSHOT = pd.Timestamp('2023-09-08')

# TARGET: what we want to predict (1=churned, 0=active)
df['is_churned'] = df['canceled_date'].notna().astype(int)

# FEATURE 1: tenure_days — how long have they been subscribed?
# Longer tenure = usually less likely to churn
df['tenure_days'] = (
    df['canceled_date'].fillna(SNAPSHOT) - df['created_date']
).dt.days

# FEATURE 2: log_tenure — compressed version of tenure_days
# log() squashes large numbers so extreme values (300+ days) 
# don't overwhelm the model. np.log1p = log(x+1) avoids log(0) error.
df['log_tenure'] = np.log1p(df['tenure_days'])

# FEATURE 3: is_paid — did they actually pay? (1=Yes, 0=No)
# Unpaid subscribers churn at 92.5% — our strongest predictor
df['is_paid'] = (df['was_subscription_paid'] == 'Yes').astype(int)

# FEATURE 4: signup_month — which month of year did they sign up?
df['signup_month'] = df['created_date'].dt.month

# FEATURE 5: signup_dow — what day of week did they sign up? (0=Mon, 6=Sun)
df['signup_dow'] = df['created_date'].dt.dayofweek

# FEATURE 6: signup_day — what day of month did they sign up? (1-31)
df['signup_day'] = df['created_date'].dt.day

FEATURES = ['tenure_days', 'log_tenure', 'is_paid',
            'signup_month', 'signup_dow', 'signup_day']
TARGET   = 'is_churned'

print('Features created:', FEATURES)
print()
print('Correlation of each feature with churn (closer to -1 or +1 = stronger):')
for f in FEATURES:
    corr = df[f].corr(df[TARGET])
    sign = 'more churn' if corr > 0 else 'less churn'
    print(f'  {f:<20}: {corr:+.3f}  (higher value → {sign})')

In [ ]:
# ── CELL 3 ── Split data into train and test sets ────────────────────────────
# WHY SPLIT?
# We train the model on 80% of data, test on the other 20%.
# The 20% test set is data the model NEVER saw during training.
# This tells us how well it performs on brand new subscribers.
#
# stratify=y → both sets have the same 65.3% churn rate (fair split)
# random_state=42 → same split every time you run this

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set: {len(X_train):,} subscribers')
print(f'Test set    : {len(X_test):,} subscribers')
print(f'Churn rate in train: {y_train.mean()*100:.1f}%')
print(f'Churn rate in test : {y_test.mean()*100:.1f}%')
print()
print('Both churn rates should be ~65.3% — stratify worked correctly.')

In [ ]:
# ── CELL 4 ── Train the Random Forest model ──────────────────────────────────
# WHAT IS RANDOM FOREST?
# 100 decision trees, each trained on slightly different data.
# Final prediction = majority vote of all 100 trees.
# More robust than a single tree — less likely to overfit.
#
# class_weight='balanced' → adjusts for the fact that we have
# 2004 churned vs 1065 active. Without this, the model might
# lazily predict "churned" every time and still be 65% accurate.

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print('Training Random Forest... (may take 10-20 seconds)')
rf_model.fit(X_train, y_train)
print('Training complete!')

# Predictions on the test set
rf_pred  = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)

# Train Logistic Regression for comparison
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
lr_auc   = roc_auc_score(y_test, lr_proba)

print()
print('=== MODEL RESULTS ===')
print(f'Random Forest  AUC-ROC: {rf_auc:.3f}  ← use this number in your README')
print(f'Logistic Reg.  AUC-ROC: {lr_auc:.3f}')
print()
print('AUC GUIDE: 0.5=random guess | 0.7=ok | 0.8=good | 0.9=excellent')

In [ ]:
# ── CELL 5 ── Full evaluation report ─────────────────────────────────────────
# WHAT THESE NUMBERS MEAN (for interviews):
# Precision: Of subscribers we predicted would churn, what % actually did?
# Recall:    Of all subscribers who actually churned, what % did we catch?
# F1 Score:  Balance between precision and recall. Higher = better.

print('=== CLASSIFICATION REPORT ===')
print(classification_report(
    y_test, rf_pred,
    target_names=['Active (0)', 'Churned (1)']
))

In [ ]:
# ── CELL 6 ── Feature importance chart ───────────────────────────────────────
# MOST IMPORTANT CHART for interviews!
# This answers: "What actually CAUSES subscribers to churn?"
# Higher importance = the model relied on this column more.

feat_imp = pd.Series(
    rf_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#D85A30' if x > feat_imp.median() else '#7F77DD'
          for x in feat_imp.values]

bars = ax.barh(feat_imp.index, feat_imp.values, color=colors, edgecolor='white')

for bar, val in zip(bars, feat_imp.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_title(
    'What Drives Churn? — Random Forest Feature Importance\n'
    'Higher score = stronger predictor of churn',
    fontsize=12, fontweight='bold', pad=15
)
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_xlim(0, feat_imp.max() * 1.2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'chart7_feature_importance.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('Feature importance chart saved!')
print()
print('=== FEATURE RANKING (for your interview answer) ===')
for feat, score in feat_imp.sort_values(ascending=False).items():
    print(f'  {feat:<20}: {score:.4f}')

In [ ]:
# ── CELL 7 ── Confusion matrix ───────────────────────────────────────────────
# Confusion matrix shows how many predictions were right and wrong.
# Top-left  = correctly said Active   (True Negative)
# Top-right = wrongly said Churned    (False Positive)
# Bot-left  = wrongly said Active     (False Negative) ← most costly miss
# Bot-right = correctly said Churned  (True Positive)

cm = confusion_matrix(y_test, rf_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
    xticklabels=['Predicted: Active', 'Predicted: Churned'],
    yticklabels=['Actual: Active', 'Actual: Churned']
)
ax.set_title(f'Confusion Matrix — AUC-ROC: {rf_auc:.3f}',
             fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'chart8_confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Correctly predicted Active  (TN): {tn}')
print(f'Wrongly predicted Churned   (FP): {fp}')
print(f'Missed churners             (FN): {fn}  ← these are costly misses')
print(f'Correctly predicted Churned (TP): {tp}')

In [ ]:
# ── CELL 8 ── Score all ACTIVE subscribers ───────────────────────────────────
# This is the REAL business value of the model.
# We apply the trained model to every currently active subscriber
# and give each one a churn probability score (0% to 100%).
# Business team uses this list to reach out to high-risk customers.

active_df = df[df['is_churned'] == 0].copy()
print(f'Active subscribers to score: {len(active_df):,}')

# Predict churn probability for every active subscriber
# predict_proba returns [[prob_active, prob_churn]] for each row
# [:, 1] takes the churn probability column
active_df['churn_probability'] = rf_model.predict_proba(
    active_df[FEATURES]
)[:, 1].round(3)

# Assign risk tiers
active_df['risk_tier'] = pd.cut(
    active_df['churn_probability'],
    bins=[0, 0.40, 0.65, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk'],
    include_lowest=True
)

# Summary table
summary = active_df.groupby('risk_tier', observed=True).agg(
    subscribers       = ('customer_id', 'count'),
    monthly_revenue   = ('subscription_cost', 'sum'),
    avg_tenure_days   = ('tenure_days', 'mean'),
    avg_churn_prob    = ('churn_probability', 'mean')
).round(1)

summary['revenue_at_risk'] = (
    summary['monthly_revenue'] * summary['avg_churn_prob']
).round(0).astype(int)

print()
print('=== ACTIVE SUBSCRIBER RISK BREAKDOWN ===')
print(summary.to_string())
print()
high = summary.loc['High Risk', 'subscribers']
rev  = summary.loc['High Risk', 'monthly_revenue']
print(f'High risk count   : {high} subscribers')
print(f'Revenue at risk   : ${rev:,}/month')
print(f'Recommended action: Contact these {high} subscribers with a retention offer')

In [ ]:
# ── CELL 9 ── Risk tier charts ────────────────────────────────────────────────

tier_order  = ['Low Risk', 'Medium Risk', 'High Risk']
tier_counts = active_df['risk_tier'].value_counts()
counts      = [tier_counts.get(t, 0) for t in tier_order]
colors      = ['#1D9E75', '#EF9F27', '#D85A30']

summary_ordered = summary.reindex(tier_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: donut chart
wedges, texts, autotexts = axes[0].pie(
    counts, labels=tier_order, colors=colors,
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55)
)
for t in autotexts:
    t.set_fontsize(11)
    t.set_fontweight('bold')
axes[0].set_title('Active Subscribers by Risk Tier', fontweight='bold', pad=15)

# Right: revenue at risk bar chart
rev_vals = summary_ordered['revenue_at_risk'].values
bars = axes[1].bar(tier_order, rev_vals, color=colors, edgecolor='white')
for bar, val in zip(bars, rev_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 30,
                 f'${val:,}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('Monthly Revenue at Risk by Tier', fontweight='bold', pad=15)
axes[1].set_ylabel('Revenue at Risk ($)')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('MavenFlix — Subscriber Churn Risk Profile',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'chart9_risk_profile.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Risk profile chart saved!')

In [ ]:
# ── CELL 10 ── Export risk scores CSV for Power BI ───────────────────────────

save_path = os.path.join(OUTPUT_DIR, 'churn_risk_scores.csv')
active_df[['customer_id', 'created_date', 'tenure_days', 'is_paid',
           'subscription_cost', 'churn_probability', 'risk_tier']]    .to_csv(save_path, index=False)

print(f'Saved: {save_path}')
print(f'Rows  : {len(active_df):,}')
print()
print('TOP 10 HIGHEST RISK SUBSCRIBERS:')
top10 = active_df.nlargest(10, 'churn_probability')[
    ['customer_id', 'tenure_days', 'is_paid', 'churn_probability', 'risk_tier']]
print(top10.to_string(index=False))
print()
print('=== NOTEBOOK 3 COMPLETE ===')
print('Next: Open 04_revenue_forecast.ipynb')